In [5]:
from datasets import load_dataset

ag_news = load_dataset("fancyzhx/ag_news")
print(ag_news["train"].features["label"].names)

['World', 'Sports', 'Business', 'Sci/Tech']


In [6]:
from gensim.models import Word2Vec

#chunks de 5000 
sentences = [line.split() for line in ag_news["train"]["text"][:5000]]
w2v = Word2Vec(
    sentences = sentences,
    vector_size = 100, #pregutnar si ese tamanio de vector es esencial
    window = 5,
    sg = 1, #para usar skip gram
    negative = 5,
    ns_exponent = 0.75,
    epochs = 10, 
    workers = 4, #numero de hilos que trabajan en paralelo
)

#TODO: investigar graficas para poner las asociaciones entre todos los
      #vectores o incluso pca

In [ ]:
print(w2v.wv.most_similar("company", topn=5))
# print(w2v.wv.similarity("company", "corporation"))

Parte de redes neuronales

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

counter = Counter()
for text in ag_news["train"]["text"]:
    counter.update(tokenize(text))

vocab = ["<pad>", "unk"] + [w for w,c in counter.most_common(20_000)]
w2i = {w: i for i, w in enumerate(vocab)}
PAD_ID, UNK_ID = 0,1

def encode(text, max_len=64):
    ids = [w2i.get(t, UNK_ID) for t in tokenize(text)][:max_len] #el segundo 
    #valor UNK_ID es por defecto
    ids += [PAD_ID] * (max_len - len(ids)) #rellenamos lo demas con paddings
    return ids


#definimos nuestros dataloaders
class AGNewsDataset(Dataset):
    def __init__(self, split):
        self.texts = split["text"]
        self.labels = split["label"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(encode(self.texts[idx])), torch.tensor(self.labels[idx])

train_loader = DataLoader(AGNewsDataset(ag_news["train"]), batch_size=64, shuffle=True)
val_loader = DataLoader(AGNewsDataset(ag_news["test"]), batch_size=128)

#hacemos nuestra red neuronal

class FeedforwardClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, d_hidden=128, n_classes=4, pad_id=0):
        super().__init__()
        self.E = nn.Embedding(vocab_size, d_model, padding_idx=pad_id) #aqui generamos una 
        #matriz donde cada fila representa un token con un vector aleatorio que se ira modificando
        self.W  = nn.Linear(d_model, d_hidden)
        self.U = nn.Linear(d_hidden, n_classes)
        self.dropout = nn.Dropout(0.3)


    def forward(self, token_ids):
        mask = (token_ids != PAD_ID).unsqueeze(-1)

        e = self.E(token_ids)
        x = (e*mask).sum(1) / mask.sum(1).clamp(min=1) #mean pooling
        h = torch.tanh(self.W(x))
        h = self.dropout(h)
        z = self.U(h)
        return z


device = "cuda" if torch.cuda.is_available() else "cpu"
model = FeedforwardClassifier(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x,y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    model.train()
    return correct / total


for epoch in range(3):
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()                    # limpia gradientes del paso anterior
        logits = model(x)                          # forward pass
        loss = F.cross_entropy(logits, y)           # Ec. 6.28
        loss.backward()                              # backward pass (Sec. 6.6.4)
        optimizer.step()                              # actualiza W, U, E (Ec. 6.20-tipo update)
    print(f"epoch {epoch}  val_acc={evaluate(val_loader):.3f}")

redes feedforward